# Lab10 part 2: Conversational AI with Retrieval-Augmented Generation (RAG)

## Introduction

In this part of the lab, we extend the Retrieval-Augmented Generation (RAG) framework to build a **Conversational AI system**. Unlike the previous section, where each query was treated independently, here the system will maintain **conversation memory** to provide context-aware responses.

### How Conversational RAG Works
1. **Document Retrieval**: As before, the system retrieves relevant documents for the current query.
2. **Conversation Memory**: The system tracks the conversation's history, allowing responses to build on prior queries and answers.
3. **LLM Integration**: The retrieved documents and conversation history are used as input to the LLM for generating responses.

Maintaining conversation memory can improve the user experience by enabling continuity, coherence, and context-awareness.

---

## Exercise 1: Building Conversational RAG with Memory

### Objective
Extend the RAG pipeline to maintain a memory of previous queries and responses while answering user queries.

### Instructions
1. Modify the RAG pipeline to include a **memory mechanism** that:
   - Stores the last `N` queries and responses (e.g., `N=3`).
   - Appends this memory to the input prompt for the LLM.
2. Update the prompt to include:
   - The current query.
   - Relevant retrieved context.
   - Conversation memory (e.g., a list of recent exchanges).
3. Use the same models as before:
   - Document Encoder: Sentence transformer.
   - Generators: Llama 3.2 (3B), Gemma 3 (4B).
4. Experiment with memory size (`N`) and prompt formatting to balance coherence and performance.

### Tasks
- Test the conversational RAG by asking a series of related queries.
- Analyze how the responses differ when memory size (`N`) changes.
- Experiment with prompt design to ensure the system sticks to retrieved context and conversation history.


## Exercise 2: Selective Retrieval from Conversational History

### Objective
Instead of appending the entire conversation history to the prompt, implement a system that saves the full history in a file and retrieves only the relevant exchanges for a given query. This ensures that the context is concise and relevant, improving efficiency and coherence.

---

### Instructions

1. **Save and Maintain Conversation History**:  
   - Save the complete conversation history (queries and responses) into a structured file (e.g., JSON or CSV).
   - Ensure the file format includes:
     - **User Query**: The input from the user.
     - **LLM Response**: The generated response.
     - **Timestamps**: To maintain chronological order and for debugging purposes.

2. **Combine Retrieval Sources**:  
   - When a new query is received:
     - Retrieve **immediate previous exchanges** to maintain conversational flow.
     - Use a **retriever model** (e.g., Sentence Transformer) to identify the top `k` most similar exchanges from the saved history file.
     - Retrieve relevant sentences from the **knowledge base** using the same retriever.

3. **Prompt Construction**:  
   - Construct a concise and informative prompt for the LLM by including:
     - Relevant parts of the **previous conversation**.
     - The **top `k` exchanges** from the saved history file.
     - Relevant sentences from the **knowledge base**.
     - The **current query**.

4. **System Testing**:  
   - Use the same models as in previous exercises:
     - Document Encoder: Sentence Transformer.
     - Generators: Llama 3.2 (3B), Gemma 3 (4B).
   - Experiment with retrieval parameters such as:
     - `k` (number of most similar exchanges to retrieve from history).
     - Scoring threshold for relevance in the retriever model.

---

## Exercise 3 (extra): Adding Multi-Turn PDF Knowledge Base

### Objective
Integrate the Conversational RAG system with a PDF knowledge base, enabling it to handle multi-turn conversations grounded in external documents.

### Instructions
1. Prepare a folder (`input_folder`) with PDFs as in the previous lab.
2. Extend the preprocessing pipeline to:
   - Extract text, tokenize sentences, and create embeddings for each PDF.
   - Store embeddings in a format optimized for retrieval during multi-turn conversations.
3. Update the inference pipeline to:
   - Match each query with the PDF embeddings.
   - Incorporate conversation memory into the context for the LLM.
4. Test the system by asking a series of related queries about the PDF content.

### Tasks
- Test the multi-turn Conversational RAG system with various PDF files.
- Analyze how memory affects the system's ability to provide coherent, context-aware responses.



In [ ]:
# -*- coding: utf-8 -*-
"""NLP_Lab10_Part1_Updated.ipynb

# Lab10 Part 1: Retrieval-Augmented Generation (RAG)

## Introduction

In this lab, we will explore Retrieval-Augmented Generation (RAG), a method that enhances
Large Language Models (LLMs) by combining them with a retrieval mechanism. RAG retrieves
relevant documents from an external knowledge base to provide context for answering user queries.
This approach mitigates the issues of hallucination and outdated knowledge by grounding
responses in retrieved documents.

### How RAG Works
1. **Document Encoder**: Converts documents into embeddings, enabling efficient similarity search.
2. **Retriever**: Matches user queries to the most relevant documents based on embeddings.
3. **Generator (LLM)**: Uses the retrieved documents to answer queries, focusing on the provided context.

By the end of this lab, you will:
- Use a sentence transformer as a document encoder.
- Experiment with different LLMs (Llama 3.2, Gemma 2, Qwen 2.5, Phi-3).
- Evaluate model performance using BLEU scores.
- Implement a complete RAG pipeline.

---

## Installation
"""

# ============================================================================
# SECTION 1: INSTALLATION
# ============================================================================

# Install PyTorch with GPU support (CUDA 12.1 for modern GPUs like T4, A100, V100)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install core libraries with specific versions for reproducibility
!pip install transformers         # Hugging Face transformers for LLMs
!pip install sentence-transformers  # For encoding text into embeddings
!pip install nltk                   # Natural Language Toolkit for BLEU scores
!pip install numpy               # Numerical operations
!pip install accelerate           # Efficient model loading and memory management
!pip install pypdf

# Download required NLTK data for tokenization and BLEU score calculation
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, util
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Load Models ---
print("Loading models...")
retriever = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
tokenizer = AutoTokenizer.from_pretrained("unsloth/Llama-3.2-3B-Instruct")
generator = AutoModelForCausalLM.from_pretrained("unsloth/Llama-3.2-3B-Instruct", torch_dtype=torch.float16, device_map="auto")

# --- Knowledge Base ---
knowledge_base = [
    "The capital of France is Paris.",
    "The Eiffel Tower was constructed in 1889.",
    "Python is a high-level programming language.",
    "RAG stands for Retrieval-Augmented Generation."
]
kb_embeddings = retriever.encode(knowledge_base, convert_to_tensor=True)

MEMORY_SIZE = 3
chat_history = []  # List of tuples: (user_query, ai_response)

# --- 1. MEMORY FORMATTING ---

def format_history(history):
    """
    Goal: Prepare the conversation history so the LLM remembers previous turns.
    Input: history (list of tuples) -> The global chat_history list.
    Output: formatted (string) -> A single string representing the recent dialogue.
    """
    # TODO: 1.1. Initialize an empty string variable named 'formatted'.

    # TODO: 1.2. Slice the 'history' list to only include the last 'MEMORY_SIZE' items.
    # Instruction: This prevents the prompt from growing too long and exceeding the model's token limit.

    # TODO: 1.3. Iterate through the sliced history.
    # Instruction: For each (user_text, ai_text) pair, append them to 'formatted'
    # in this style: "User: {user_text}\nAssistant: {ai_text}\n"

    # TODO: 1.4. Return the 'formatted' string.
    pass


# --- 2. RAG GENERATION LOGIC ---

def chat_generate(query):
    """
    Goal: Implement the "Retrieve-Augment-Generate" flow with conversational memory.
    Input: query (string) -> The user's new question.
    Output: response (string) -> The final answer generated by the model.
    """

    # --- SUB-STEP A: RETRIEVAL ---
    # Goal: Find the most relevant facts from the Knowledge Base (KB).

    # TODO: 2.1. Encode the 'query' string into a vector using 'retriever.encode'.
    # Note: Use convert_to_tensor=True to keep it on the GPU/CPU.

    # TODO: 2.2. Compute similarity between your query vector and 'kb_embeddings'.
    # Hint: Use 'util.cos_sim' from the sentence_transformers library.

    # TODO: 2.3. Extract the top 2 indices with the highest similarity scores.
    # Hint: Use 'torch.topk(scores, k=2).indices'.

    # TODO: 2.4. Map those indices back to the 'knowledge_base' text list.
    # Instruction: Join the results into a single string called 'context' separated by newlines.


    # --- SUB-STEP B: PROMPT ENGINEERING ---
    # Goal: Combine System Rules, Context, History, and Query into a single Llama-3 prompt.

    # TODO: 2.5. Get the formatted history string by calling 'format_history(chat_history)'.

    # TODO: 2.6. Create a 'system_prompt' string.
    # Instruction: Tell the AI it is a technical expert. Command it to ONLY use
    # the provided context. Tell it to say "I don't know" if the answer isn't there.

    # TODO: 2.7. Construct 'full_prompt' using Llama-3's specific Chat Template.
    # Detailed Formatting Instructions:
    # 1. Start with "<|begin_of_text|>"
    # 2. Add System Header: "<|start_header_id|>system<|end_header_id|>\n\n{system_prompt}<|eot_id|>"
    # 3. Add User Header: "<|start_header_id|>user<|end_header_id|>\n\n"
    #    Inside this block, clearly label:
    #    "Context: {context}\n"
    #    "Chat History: {history_text}\n"
    #    "New Question: {query}<|eot_id|>"
    # 4. Add Assistant Header: "<|start_header_id|>assistant<|end_header_id|>\n\n"


    # --- SUB-STEP C: GENERATION ---
    # Goal: Run the model and clean the output.

    # TODO: 2.8. Tokenize 'full_prompt' and move the resulting 'inputs' to the 'device'.

    # TODO: 2.9. Generate tokens using 'generator.generate'.
    # Instruction: Set max_new_tokens=150 and do_sample=False (greedy decoding).

    # TODO: 2.10. Decode the output tokens.
    # CRITICAL: Slice the decoded text so you ONLY return the new text generated
    # AFTER the prompt (to avoid repeating the user's question back to them).


    # --- SUB-STEP D: MEMORY UPDATE ---
    # TODO: 2.11. Append the (query, response) as a tuple to the global 'chat_history'.

    # TODO: 2.12. Return the 'response' string.
    pass


# --- 3. THE INTERACTIVE LOOP ---

# TODO: 3.1. Create a loop that runs until the user types 'exit'.
# TODO: 3.2. Inside the loop, take user input using 'input()'.
# TODO: 3.3. Call 'chat_generate(user_input)'.
# TODO: 3.4. Print the AI's response to the console.
